# HR Employee Attrition — Exploratory Data Analysis

IBM HR Analytics dataset: patterns, visualizations, and business insights.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import check_data_quality, clean_data, load_raw_data

FIGURES_DIR = PROJECT_ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_FILE = PROJECT_ROOT / 'data' / 'WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = load_raw_data(DATA_FILE)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
quality = check_data_quality(df)
print('Missing values:', quality['total_missing'])
print('Duplicates:', quality['duplicate_rows'])
df_clean = clean_data(df)
print('After cleaning:', df_clean.shape)
df_clean.describe(include='all').T.head(15)

## 1. Attrition Distribution

**Insight:** ~16% attrition rate indicates class imbalance; retention efforts should focus on the minority leaving group.

In [ ]:
fig, ax = plt.subplots()
df_clean['Attrition'].value_counts().plot(kind='bar', ax=ax, color=['#2ECC71', '#E74C3C'])
ax.set_title('Employee Attrition Distribution')
ax.set_xlabel('Attrition')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_attrition_distribution.png', dpi=120)
plt.show()

## 2. Attrition by Department

**Insight:** Sales typically has the highest attrition; targeted retention in Sales yields the largest impact.

In [ ]:
dept = df_clean.groupby('Department')['Attrition'].apply(lambda x: (x=='Yes').mean()*100).sort_values(ascending=False)
fig, ax = plt.subplots()
dept.plot(kind='bar', ax=ax, color='#E74C3C')
ax.set_title('Attrition Rate by Department (%)')
ax.set_ylabel('Attrition Rate (%)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_attrition_by_department.png', dpi=120)
plt.show()

## 3. Attrition by Age

**Insight:** Younger age groups show higher turnover; early-career programs reduce churn risk.

In [ ]:
df_clean['AgeGroup'] = pd.cut(df_clean['Age'], bins=[17,25,35,45,55,65], labels=['18-25','26-35','36-45','46-55','56+'])
age = df_clean.groupby('AgeGroup', observed=True)['Attrition'].apply(lambda x: (x=='Yes').mean()*100)
fig, ax = plt.subplots()
age.plot(kind='bar', ax=ax, color='#F39C12')
ax.set_title('Attrition Rate by Age Group (%)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_attrition_by_age.png', dpi=120)
plt.show()

## 4. Attrition by Salary

**Insight:** Lower salary bands correlate with higher attrition; pay equity reviews in lower bands are high-ROI.

In [ ]:
df_clean['SalaryBand'] = pd.qcut(df_clean['MonthlyIncome'], 4, labels=['Low','Lower-Mid','Upper-Mid','High'], duplicates='drop')
sal = df_clean.groupby('SalaryBand', observed=True)['Attrition'].apply(lambda x: (x=='Yes').mean()*100)
fig, ax = plt.subplots()
sal.plot(kind='bar', ax=ax, color='#3498DB')
ax.set_title('Attrition Rate by Salary Band (%)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_attrition_by_salary.png', dpi=120)
plt.show()

## 5. Attrition by Overtime

**Insight:** Overtime workers leave at much higher rates; workload and burnout management are critical.

In [ ]:
ot = df_clean.groupby('OverTime')['Attrition'].apply(lambda x: (x=='Yes').mean()*100)
fig, ax = plt.subplots()
ot.plot(kind='bar', ax=ax, color=['#3498DB', '#E74C3C'])
ax.set_title('Attrition Rate by Overtime (%)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_attrition_by_overtime.png', dpi=120)
plt.show()

## 6. Attrition by Job Satisfaction

**Insight:** Lower satisfaction scores align with higher attrition; pulse surveys help early intervention.

In [ ]:
js = df_clean.groupby('JobSatisfaction')['Attrition'].apply(lambda x: (x=='Yes').mean()*100)
fig, ax = plt.subplots()
js.plot(kind='line', marker='o', ax=ax, color='#9B59B6')
ax.set_title('Attrition Rate by Job Satisfaction (1=Low, 4=High)')
ax.set_xlabel('Job Satisfaction')
ax.set_ylabel('Attrition Rate (%)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_attrition_by_job_satisfaction.png', dpi=120)
plt.show()

## 7. Correlation Heatmap

**Insight:** Income, tenure, and satisfaction variables interrelate; use ensemble models for prediction.

In [ ]:
numeric = df_clean.select_dtypes(include=np.number)
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(numeric.corr(), annot=False, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_correlation_heatmap.png', dpi=120)
plt.show()

## Summary

Key drivers: **overtime**, **low satisfaction**, **younger age**, **lower pay**, and **Sales department**. Use the trained Random Forest model in `models/` for individual risk scoring.